# Complete Guide to Model Calibration

This notebook provides a comprehensive, interactive guide to model calibration using the `incerto` library.

**What you'll learn:**
- What calibration is and why it matters
- How to measure calibration (7 metrics including smECE)
- How to visualize calibration (reliability diagrams, confidence histograms, smooth diagrams)
- 8 post-hoc calibration methods
- 4 training-time calibration methods
- Save/load calibrators for production

**Runtime:** ~3 min (MPS/CUDA), ~8 min (CPU)

**Prerequisites:** Basic PyTorch knowledge

## Setup

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# Post-hoc calibration methods
from incerto.calibration import (
    TemperatureScaling,
    VectorScaling,
    MatrixScaling,
    DirichletCalibrator,
    IsotonicRegressionCalibrator,
    HistogramBinningCalibrator,
    PlattScalingCalibrator,
    BetaCalibrator,
)

# Training-time calibration methods
from incerto.calibration import (
    LabelSmoothingLoss,
    FocalLoss,
    ConfidencePenalty,
    TemperatureAwareTraining,
)

# Metrics
from incerto.calibration import (
    ece_score,
    mce_score,
    brier_score,
    nll,
    classwise_ece,
    adaptive_ece_score,
    smooth_ece,
)

# Visualizations
from incerto.calibration import (
    plot_reliability_diagram,
    plot_confidence_histogram,
    plot_calibration_curve,
    plot_smooth_reliability_diagram,
)

from incerto.utils import ConvNet, seed_everything

seed_everything(42)

# Device selection: CUDA > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

## Part 1: Understanding Calibration

### What is calibration?

A model is **calibrated** if its predicted confidence matches empirical accuracy:
- If model predicts 80% confidence, it should be correct 80% of the time
- If model predicts 50% confidence, it should be correct 50% of the time

### Why do models become miscalibrated?

Modern neural networks tend to be **overconfident**:
- Trained to minimize cross-entropy (encourages high confidence)
- Complex models can fit training data perfectly
- No explicit penalty for overconfidence

## Part 2: Load Data and Train Model

We use **Fashion-MNIST** — a harder drop-in replacement for MNIST with the same
format (28x28 grayscale, 10 classes). Unlike MNIST where CNNs reach 99%+ accuracy
and are already well-calibrated, Fashion-MNIST (~90% accuracy) naturally produces
overconfident predictions, making it ideal for demonstrating calibration methods.

In [ ]:
# Load Fashion-MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])

train_dataset = datasets.FashionMNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST('./data', train=False, transform=transform)

# Split training data: 50k train, 10k calibration
train_size = 50000
cal_size = 10000
train_subset, cal_dataset = random_split(train_dataset, [train_size, cal_size])

# Optimized data loaders
num_workers = min(4, os.cpu_count() or 0)
pin_memory = device.type == "cuda"
loader_kwargs = dict(num_workers=num_workers, pin_memory=pin_memory, persistent_workers=num_workers > 0)

train_loader = DataLoader(train_subset, batch_size=256, shuffle=True, **loader_kwargs)
cal_loader = DataLoader(cal_dataset, batch_size=512, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False, **loader_kwargs)

print(f"Training: {len(train_subset)} | Calibration: {len(cal_dataset)} | Test: {len(test_dataset)}")
print(f"DataLoader: num_workers={num_workers}, pin_memory={pin_memory}")

In [ ]:
# Train a CNN (no dropout — typical of models that become overconfident)
model = ConvNet(num_classes=10, dropout_rate=0.0).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

model.train()
print("Training model (10 epochs on Fashion-MNIST, no dropout)...")
for epoch in range(10):
    total_loss = 0
    correct = 0
    total = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    if (epoch + 1) % 2 == 0:
        accuracy = 100. * correct / total
        print(f"  Epoch {epoch+1}: Loss = {total_loss/len(train_loader):.4f}, Accuracy = {accuracy:.2f}%")

print("Done!")

In [ ]:
# Collect logits on calibration and test sets
model.eval()

def collect_logits(mdl, loader):
    all_logits, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            logits = mdl(inputs.to(device))
            all_logits.append(logits.cpu())
            all_labels.append(labels)
    return torch.cat(all_logits), torch.cat(all_labels)

cal_logits, cal_labels = collect_logits(model, cal_loader)
test_logits, test_labels = collect_logits(model, test_loader)

print(f"Calibration set: {cal_logits.shape[0]} samples, logits shape: {cal_logits.shape}")
print(f"Test set: {test_logits.shape[0]} samples")

## Part 3: Calibration Metrics

`incerto` provides 7 calibration metrics:

| Metric | Type | Description |
|--------|------|-------------|
| `ece_score` | Binned | Expected Calibration Error |
| `mce_score` | Binned | Maximum Calibration Error |
| `adaptive_ece_score` | Adaptive binned | ECE with equal-mass bins (Nixon et al., 2019) |
| `classwise_ece` | Per-class | Average ECE across classes |
| `smooth_ece` | Kernel-smoothed | Binning-free smECE (Blasiok & Nakkiran, ICLR 2024) |
| `brier_score` | Proper scoring | MSE between predictions and one-hot labels |
| `nll` | Proper scoring | Negative log-likelihood |

In [ ]:
# Compute all calibration metrics before calibration
print("Calibration Metrics (BEFORE calibration):")
print("=" * 55)

metrics_before = {
    "ECE":          ece_score(cal_logits, cal_labels, n_bins=15),
    "MCE":          mce_score(cal_logits, cal_labels, n_bins=15),
    "Adaptive ECE": adaptive_ece_score(cal_logits, cal_labels, n_bins=15),
    "Classwise ECE": classwise_ece(cal_logits, cal_labels, n_bins=15),
    "Smooth ECE":   smooth_ece(cal_logits, cal_labels),
    "Brier Score":  brier_score(cal_logits, cal_labels),
    "NLL":          nll(cal_logits, cal_labels),
}

for name, value in metrics_before.items():
    print(f"  {name:15s}: {value:.4f}")

if metrics_before["ECE"] > 0.02:
    print(f"\nModel is overconfident (ECE = {metrics_before['ECE']:.4f}) — calibration should help")
else:
    print(f"\nModel is reasonably well calibrated (ECE = {metrics_before['ECE']:.4f})")

## Part 4: Calibration Visualizations

`incerto` provides 4 built-in visualization functions.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. Reliability diagram: confidence vs accuracy per bin
plot_reliability_diagram(cal_logits, cal_labels, n_bins=15, ax=axes[0, 0])

# 2. Confidence histogram: distribution of max softmax probabilities
plot_confidence_histogram(cal_logits, n_bins=15, ax=axes[0, 1])

# 3. Calibration curve: accuracy vs bin center
plot_calibration_curve(cal_logits, cal_labels, n_bins=15, ax=axes[1, 0])

# 4. Smooth reliability diagram (smECE, Blasiok & Nakkiran 2024)
plot_smooth_reliability_diagram(cal_logits, cal_labels, ax=axes[1, 1])

plt.tight_layout()
plt.show()

## Part 5: Post-hoc Calibration Methods

Post-hoc methods are applied **after training** on a held-out calibration set.
All follow the same API: `calibrator.fit(logits, labels)` then `calibrator.predict(logits)`.

| Method | Parameters | Best for |
|--------|-----------|----------|
| Temperature Scaling | 1 scalar | Default choice, works well with little data |
| Vector Scaling | C scalars | Per-class temperature adjustment |
| Matrix Scaling | C² + C | Maximum flexibility, needs more data |
| Dirichlet | C² + C | Matrix scaling with regularization |
| Isotonic Regression | Non-parametric | Flexible, per-class fitting |
| Histogram Binning | Non-parametric | Simple, interpretable |
| Platt Scaling | Per-class logistic | Classic approach |
| Beta Calibration | 3 params | Binary classification |

In [ ]:
# Fit all post-hoc calibrators
calibrators = {}

# 1. Temperature Scaling (recommended default)
calibrators["Temperature"] = TemperatureScaling()
calibrators["Temperature"].fit(cal_logits, cal_labels, max_iters=50)
print(f"Temperature Scaling: T = {calibrators['Temperature'].temperature.item():.4f}")

# 2. Vector Scaling (per-class temperature)
calibrators["Vector"] = VectorScaling(n_classes=10)
calibrators["Vector"].fit(cal_logits, cal_labels, max_iters=50)
print(f"Vector Scaling: {repr(calibrators['Vector'])}")

# 3. Matrix Scaling (full affine transform)
calibrators["Matrix"] = MatrixScaling(n_classes=10)
calibrators["Matrix"].fit(cal_logits, cal_labels, max_iters=50)
print("Matrix Scaling: fitted")

# 4. Dirichlet Calibration (regularized matrix scaling)
calibrators["Dirichlet"] = DirichletCalibrator(n_classes=10, mu=0.01)
calibrators["Dirichlet"].fit(cal_logits, cal_labels, max_iters=100)
print(f"Dirichlet: {repr(calibrators['Dirichlet'])}")

# 5. Isotonic Regression (non-parametric, per-class)
calibrators["Isotonic"] = IsotonicRegressionCalibrator()
calibrators["Isotonic"].fit(cal_logits, cal_labels)
print(f"Isotonic Regression: {repr(calibrators['Isotonic'])}")

# 6. Histogram Binning (non-parametric)
calibrators["Histogram"] = HistogramBinningCalibrator(n_bins=15)
calibrators["Histogram"].fit(cal_logits, cal_labels)
print(f"Histogram Binning: {repr(calibrators['Histogram'])}")

# 7. Platt Scaling (per-class logistic regression)
calibrators["Platt"] = PlattScalingCalibrator()
calibrators["Platt"].fit(cal_logits, cal_labels)
print(f"Platt Scaling: {repr(calibrators['Platt'])}")

print("\nAll calibrators fitted!")

In [ ]:
# Compare post-hoc methods on test set
print("Post-hoc Calibration Results (test set):")
print("=" * 55)

results = {}

# Uncalibrated baseline
results["Uncalibrated"] = ece_score(test_logits, test_labels, n_bins=15)

# Each calibrator
for name, cal in calibrators.items():
    calibrated = cal.predict(test_logits)
    results[name] = ece_score(calibrated.logits, test_labels, n_bins=15)

for name, ece in results.items():
    marker = " <-- best" if ece == min(results.values()) else ""
    print(f"  {name:15s}: ECE = {ece:.4f}{marker}")

best = min(results, key=results.get)
print(f"\nBest method: {best} (ECE = {results[best]:.4f})")

In [ ]:
# Visual comparison: before vs after (Temperature Scaling)
temp_calibrated = calibrators["Temperature"].predict(test_logits)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

plot_reliability_diagram(test_logits, test_labels, n_bins=15, ax=axes[0],
                         title="Before Calibration")
plot_reliability_diagram(temp_calibrated.logits, test_labels, n_bins=15, ax=axes[1],
                         title="After Temperature Scaling")

plt.tight_layout()
plt.show()

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 5))

names = list(results.keys())
values = [results[n] for n in names]
colors = ['red'] + ['steelblue'] * (len(names) - 1)

bars = ax.bar(names, values, color=colors, alpha=0.8)
ax.axhline(y=0.05, color='orange', linestyle='--', label='Well calibrated threshold')
ax.set_ylabel('ECE (lower is better)')
ax.set_title('Post-hoc Calibration Method Comparison')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

### Beta Calibration (Binary Classification)

`BetaCalibrator` is designed for binary classification. It fits the model:

$$\text{logit}(q) = a \cdot \log(p) + b \cdot \log(1-p) + c$$

where $p$ is the uncalibrated probability and $q$ is the calibrated output.

In [ ]:
# Beta calibration demo (binary)
n_binary = 500
binary_logits = torch.randn(n_binary, 2) * 2  # Overconfident binary model
binary_labels = torch.randint(0, 2, (n_binary,))

beta_cal = BetaCalibrator()
beta_cal.fit(binary_logits, binary_labels)

print(f"Beta Calibrator: {repr(beta_cal)}")
print(f"ECE before: {ece_score(binary_logits, binary_labels):.4f}")

beta_calibrated = beta_cal.predict(binary_logits)
print(f"ECE after:  {ece_score(beta_calibrated.logits, binary_labels):.4f}")

## Part 6: Training-time Calibration

These methods improve calibration **during training** by modifying the loss function.

| Method | Key idea |
|--------|----------|
| `LabelSmoothingLoss` | Softens hard labels to prevent overconfidence |
| `FocalLoss` | Down-weights easy examples, focuses on hard ones |
| `ConfidencePenalty` | Penalizes low-entropy (overconfident) predictions |
| `TemperatureAwareTraining` | Learns temperature jointly during training |

> **Caveat:** Label smoothing with `alpha=0.1` caps the target probability at ~0.91.
> If the model is genuinely >91% accurate, this forces *underconfidence* and can
> increase ECE. Lower `alpha` (0.01–0.05) or use Focal Loss for hard tasks where
> the model is already highly accurate.

In [ ]:
# Train three models with different loss functions and compare
def train_model(mdl, crit, loader, epochs=10):
    """Train a model and return final accuracy."""
    opt = torch.optim.Adam(mdl.parameters(), lr=0.001)
    mdl.train()
    for epoch in range(epochs):
        correct, total = 0, 0
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            opt.zero_grad()
            outputs = mdl(inputs)
            loss = crit(outputs, labels)
            loss.backward()
            opt.step()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return 100. * correct / total


# 1. Label Smoothing
seed_everything(42)
model_ls = ConvNet(num_classes=10, dropout_rate=0.0).to(device)
acc_ls = train_model(model_ls, LabelSmoothingLoss(smoothing=0.1), train_loader)
print(f"Label Smoothing (alpha=0.1):    accuracy = {acc_ls:.2f}%")

# 2. Focal Loss
seed_everything(42)
model_fl = ConvNet(num_classes=10, dropout_rate=0.0).to(device)
acc_fl = train_model(model_fl, FocalLoss(gamma=2.0), train_loader)
print(f"Focal Loss (gamma=2.0):         accuracy = {acc_fl:.2f}%")

# 3. Confidence Penalty
seed_everything(42)
model_cp = ConvNet(num_classes=10, dropout_rate=0.0).to(device)
acc_cp = train_model(model_cp, ConfidencePenalty(beta=0.1), train_loader)
print(f"Confidence Penalty (beta=0.1):  accuracy = {acc_cp:.2f}%")

In [ ]:
# Compare calibration of training-time methods on test set
training_models = {
    "Standard CE": model,
    "Label Smoothing": model_ls,
    "Focal Loss": model_fl,
    "Confidence Penalty": model_cp,
}

training_results = {}
for name, m in training_models.items():
    m.eval()
    logits_m, labels_m = collect_logits(m, test_loader)
    training_results[name] = {
        "ECE": ece_score(logits_m, labels_m, n_bins=15),
        "smECE": smooth_ece(logits_m, labels_m),
        "Brier": brier_score(logits_m, labels_m),
    }

print("Training-time Method Comparison (test set):")
print("=" * 55)
print(f"  {'Method':22s} {'ECE':>8s} {'smECE':>8s} {'Brier':>8s}")
print(f"  {'-'*22} {'-'*8} {'-'*8} {'-'*8}")
for name, m in training_results.items():
    print(f"  {name:22s} {m['ECE']:8.4f} {m['smECE']:8.4f} {m['Brier']:8.4f}")

### Temperature-Aware Training

`TemperatureAwareTraining` wraps any backbone model and learns a temperature parameter jointly during training.

In [ ]:
seed_everything(42)
backbone = ConvNet(num_classes=10, dropout_rate=0.0).to(device)
model_tat = TemperatureAwareTraining(backbone, init_temp=1.5).to(device)

opt = torch.optim.Adam(model_tat.parameters(), lr=0.001)
model_tat.train()

for epoch in range(10):
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        opt.zero_grad()
        outputs = model_tat(inputs)
        loss = F.cross_entropy(outputs, labels)
        loss.backward()
        opt.step()

print(f"Learned temperature: {model_tat.temperature.item():.4f}")

# Evaluate
model_tat.eval()
tat_logits, tat_labels = collect_logits(model_tat, test_loader)

print(f"ECE:   {ece_score(tat_logits, tat_labels, n_bins=15):.4f}")
print(f"smECE: {smooth_ece(tat_logits, tat_labels):.4f}")

## Part 7: Save and Load Calibrators

All calibrators support `save()` / `load()` for production deployment.

In [ ]:
import tempfile
import os

with tempfile.TemporaryDirectory() as tmpdir:
    # Save
    path = os.path.join(tmpdir, "temperature_calibrator.pt")
    calibrators["Temperature"].save(path)
    print(f"Saved ({os.path.getsize(path)} bytes)")

    # Load
    loaded = TemperatureScaling()
    loaded.load_state_dict(torch.load(path, weights_only=True))

    # Verify predictions match
    original_probs = calibrators["Temperature"].predict(test_logits[:5]).probs
    loaded_probs = loaded.predict(test_logits[:5]).probs
    print(f"Predictions match: {torch.allclose(original_probs, loaded_probs)}")
    print(f"Loaded temperature: {loaded.temperature.item():.4f}")

## Part 8: Production Deployment

Here's how to use calibration in production:

In [ ]:
def predict_with_calibration(model, calibrator, inputs):
    """
    Production inference with calibration.
    
    Args:
        model: Trained model
        calibrator: Fitted calibrator (e.g., TemperatureScaling)
        inputs: Input batch
    
    Returns:
        predictions: Class predictions
        calibrated_probs: Calibrated probabilities
    """
    model.eval()
    with torch.no_grad():
        logits = model(inputs)
        calibrated_dist = calibrator.predict(logits)
        calibrated_probs = calibrated_dist.probs
        predictions = calibrated_probs.argmax(dim=-1)
    return predictions, calibrated_probs

# Example usage
example_batch, example_labels = next(iter(test_loader))
example_batch = example_batch[:5].to(device)

preds, probs = predict_with_calibration(model, calibrators["Temperature"], example_batch)

CLASS_NAMES = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

print("Production Example:")
print("=" * 65)
for i in range(5):
    pred_class = preds[i].item()
    confidence = probs[i, pred_class].item()
    true_class = example_labels[i].item()
    status = "correct" if pred_class == true_class else "WRONG"
    print(f"  {CLASS_NAMES[pred_class]:10s} ({confidence:.1%} conf) | true: {CLASS_NAMES[true_class]:10s} [{status}]")

## Summary

### Best Practices

1. **Start with Temperature Scaling** -- simplest and most effective
2. **Use a separate calibration set** -- never calibrate on training data
3. **Need 500+ samples minimum** for temperature scaling
4. **Evaluate on a held-out test set** -- calibration can overfit
5. **Save calibrator alongside model** for deployment
6. **Use smECE for reporting** -- binning-free, consistent, principled

### When to Use What

| Situation | Recommended |
|-----------|-------------|
| Default / quick fix | `TemperatureScaling` |
| Per-class differences | `VectorScaling` or `IsotonicRegressionCalibrator` |
| Lots of calibration data | `MatrixScaling` or `DirichletCalibrator` |
| Binary classification | `BetaCalibrator` or `PlattScalingCalibrator` |
| During training | `LabelSmoothingLoss` or `FocalLoss` |
| Reporting calibration error | `smooth_ece` (smECE) |
